In [ ]:
# Cell 1: Environment Setup & Installations
from google.colab import drive
drive.mount('/content/drive')

import sys
!{sys.executable} -m pip install -q pysodmetrics scipy timm git+https://github.com/facebookresearch/segment-anything.git
print("Packages successfully installed and Drive mounted.")

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 12.7 MB/s eta 0:00:00
Packages successfully installed and Drive mounted.


In [ ]:
# Cell 2B: Download using the shared Google Drive ID
!pip install -q --upgrade gdown
!mkdir -p /content/dataset
%cd /content/dataset
!gdown 1vRYAie0JcNStcSwagmCq55eirGyMYGm5
!unzip -q *.zip
!rm -f *.zip
%cd /content
print("Dataset downloaded and extracted via gdown!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
/content/dataset
Downloading...
From (original): https://drive.google.com/uc?id=1vRYAie0JcNStcSwagmCq55eirGyMYGm5
From (redirected): https://drive.google.com/uc?id=1vRYAie0JcNStcSwagmCq55eirGyMYGm5&confirm=t&uuid=05130bbc-0969-4f5d-9762-17c0d81fa503
To: /content/dataset/COD10K-v3.zip
100% 2.42G/2.42G [00:48<00:00, 49.6MB/s]
/content
Dataset downloaded and extracted via gdown!


In [ ]:
# Cell 3: Dynamically locate test directories
import os
import glob

test_img_dir = None
test_mask_dir = None

for root, dirs, files in os.walk("/content/dataset"):
    # Look for the Test image folder
    if root.endswith(os.path.join("Test", "Image")) or root.endswith("Test/Image"):
        test_img_dir = root
    # Look for the Test ground truth mask folder
    if root.endswith(os.path.join("Test", "GT_Object")) or root.endswith("Test/GT_Object"):
        test_mask_dir = root

print(f"Found test_img_dir  : {test_img_dir}")
print(f"Found test_mask_dir : {test_mask_dir}")

assert test_img_dir is not None, "Error: Could not locate Test/Image folder under /content/dataset!"
assert test_mask_dir is not None, "Error: Could not locate Test/GT_Object folder under /content/dataset!"

test_imgs = glob.glob(os.path.join(test_img_dir, "*.[jJ][pP][gG]"))
print(f"Total test images detected: {len(test_imgs)}")

Found test_img_dir  : /content/dataset/COD10K-v3/Test/Image
Found test_mask_dir : /content/dataset/COD10K-v3/Test/GT_Object
Total test images detected: 4000


In [ ]:
# Cell 3: Dynamically locate test directories
import os
import glob

test_img_dir = None
test_mask_dir = None

for root, dirs, files in os.walk("/content/dataset"):
    # Look for the Test image folder
    if root.endswith(os.path.join("Test", "Image")) or root.endswith("Test/Image"):
        test_img_dir = root
    # Look for the Test ground truth mask folder
    if root.endswith(os.path.join("Test", "GT_Object")) or root.endswith("Test/GT_Object"):
        test_mask_dir = root

print(f"Found test_img_dir  : {test_img_dir}")
print(f"Found test_mask_dir : {test_mask_dir}")

assert test_img_dir is not None, "Error: Could not locate Test/Image folder under /content/dataset!"
assert test_mask_dir is not None, "Error: Could not locate Test/GT_Object folder under /content/dataset!"

test_imgs = glob.glob(os.path.join(test_img_dir, "*.[jJ][pP][gG]"))
print(f"Total test images detected: {len(test_imgs)}")

Found test_img_dir  : /content/dataset/COD10K-v3/Test/Image
Found test_mask_dir : /content/dataset/COD10K-v3/Test/GT_Object
Total test images detected: 4000


In [ ]:
# Cell 1: Exact Architecture Definition & Checkpoint Loader
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from segment_anything import sam_model_registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = "/content/drive/MyDrive/COD_Checkpoints/edge_sam_best.pth"
sam_base_path = "/content/sam_vit_b_01ec64.pth"

# 1. Exact Sobel Edge Detector (uses sobel_x and sobel_y)
class SobelEdgeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, x):
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        gx = F.conv2d(gray, self.sobel_x, padding=1)
        gy = F.conv2d(gray, self.sobel_y, padding=1)
        edges = torch.sqrt(gx**2 + gy**2 + 1e-6)
        return (edges - edges.min()) / (edges.max() - edges.min() + 1e-6)

# 2. Exact Edge Guided Adapter (uses single Conv2d edge_proj)
class EdgeGuidedAdapter(nn.Module):
    def __init__(self, in_dim=256, bottleneck=64):
        super().__init__()
        self.down = nn.Conv2d(in_dim, bottleneck, kernel_size=1)
        self.edge_proj = nn.Conv2d(1, bottleneck, kernel_size=3, padding=1)
        self.act = nn.GELU()
        self.up = nn.Conv2d(bottleneck, in_dim, kernel_size=1)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, feat, edge_map):
        edge_down = F.interpolate(edge_map, size=feat.shape[-2:], mode="bilinear", align_corners=False)
        fused = self.act(self.down(feat) + self.edge_proj(edge_down))
        return feat + self.up(fused)

# 3. Exact Adapted SAM Decoder
class AdaptedSAMDecoder(nn.Module):
    def __init__(self, sam_checkpoint):
        super().__init__()
        base_sam = sam_model_registry["vit_b"](checkpoint=sam_checkpoint)
        self.edge_detector = SobelEdgeDetector()
        self.adapter = EdgeGuidedAdapter(in_dim=256, bottleneck=64)
        self.prompt_encoder = base_sam.prompt_encoder
        self.mask_decoder = base_sam.mask_decoder

        for p in self.prompt_encoder.parameters():
            p.requires_grad = False
        for p in self.mask_decoder.parameters():
            p.requires_grad = True

    def forward(self, imgs, cached_feats):
        edge_map = self.edge_detector(imgs)
        refined_feats = self.adapter(cached_feats, edge_map)

        sparse_pe, dense_pe = self.prompt_encoder(points=None, boxes=None, masks=None)
        low_res_masks, _ = self.mask_decoder(
            image_embeddings=refined_feats,
            image_pe=self.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_pe,
            dense_prompt_embeddings=dense_pe,
            multimask_output=False,
        )
        pred_masks = F.interpolate(low_res_masks, size=(1024, 1024), mode="bilinear", align_corners=False)
        return pred_masks, edge_map

# 4. Instantiate and load weights safely
model = AdaptedSAMDecoder(sam_base_path).to(device)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

model.load_state_dict(state_dict, strict=True)
model.eval()
print(f"[✓] Validation Success: Model weights loaded with 0 key errors from {ckpt_path}")

[✓] Validation Success: Model weights loaded with 0 key errors from /content/drive/MyDrive/COD_Checkpoints/edge_sam_best.pth


In [ ]:
# Cell 2: Post-Processing and Self-Prompting Functions
import numpy as np
import scipy.ndimage as ndi

def post_process_mask(prob_map, threshold=0.5, min_size=150):
    """Fills enclosed cavities and removes disconnected noise."""
    binary_mask = (prob_map >= threshold).astype(bool)
    filled = ndi.binary_fill_holes(binary_mask)
    labeled, num_features = ndi.label(filled)
    if num_features > 0:
        sizes = ndi.sum(filled, labeled, range(num_features + 1))
        mask_sizes = sizes < min_size
        remove_pixel = mask_sizes[labeled]
        filled[remove_pixel] = False
    return filled.astype(np.float32)

def self_prompted_forward(model, img, feat, pad_ratio=0.05, threshold=0.5):
    """
    Pass 1: Generates coarse prediction.
    Pass 2: Computes a padded bounding box and decodes via prompt encoder.
    """
    with torch.no_grad():
        coarse_logits, edge_map = model(img, feat)
        prob = torch.sigmoid(coarse_logits).squeeze()

    mask_indices = torch.nonzero(prob > threshold)

    # If no foreground or tiny speckles detected, fallback to unprompted prediction
    if mask_indices.shape[0] < 80:
        return torch.sigmoid(coarse_logits).squeeze().cpu().numpy()

    # Pass 2: Extract bounding box (y_min, x_min, y_max, x_max)
    y_min = mask_indices[:, 0].min().item()
    x_min = mask_indices[:, 1].min().item()
    y_max = mask_indices[:, 0].max().item()
    x_max = mask_indices[:, 1].max().item()

    h, w = y_max - y_min, x_max - x_min
    y_min = max(0, y_min - h * pad_ratio)
    x_min = max(0, x_min - w * pad_ratio)
    y_max = min(1024, y_max + h * pad_ratio)
    x_max = min(1024, x_max + w * pad_ratio)

    # SAM prompt encoder expects box tensor formatted as [x1, y1, x2, y2]
    box = torch.tensor([[x_min, y_min, x_max, y_max]], dtype=torch.float32, device=img.device)

    with torch.no_grad():
        refined_feats = model.adapter(feat, edge_map)
        sparse_pe, dense_pe = model.prompt_encoder(points=None, boxes=box, masks=None)
        low_res, _ = model.mask_decoder(
            image_embeddings=refined_feats,
            image_pe=model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_pe,
            dense_prompt_embeddings=dense_pe,
            multimask_output=False,
        )
        refined_mask = F.interpolate(low_res, size=(1024, 1024), mode="bilinear", align_corners=False)
        return torch.sigmoid(refined_mask).squeeze().cpu().numpy()

print("[✓] Inference functions defined successfully.")

[✓] Inference functions defined successfully.


In [ ]:
import os
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Locate Test Image and Mask directories
test_img_dir = None
test_mask_dir = None

for root, dirs, files in os.walk("/content/dataset"):
    if root.endswith(os.path.join("Test", "Image")) or root.endswith("Test/Image"):
        test_img_dir = root
    if root.endswith(os.path.join("Test", "GT_Object")) or root.endswith("Test/GT_Object"):
        test_mask_dir = root

if test_img_dir is None:
    test_img_dir = "/content/dataset/COD10K-v3/Test/Image"
if test_mask_dir is None:
    test_mask_dir = "/content/dataset/COD10K-v3/Test/GT_Object"

test_cache_dir = "/content/cached_feats_test"

class CachedCODDataset(Dataset):
    def __init__(self, img_dir, mask_dir, cache_dir, img_size=1024):
        self.img_paths = sorted(
            glob.glob(os.path.join(img_dir, "*.[jJ][pP][gG]")) +
            glob.glob(os.path.join(img_dir, "*.[pP][nN][gG]")) +
            glob.glob(os.path.join(img_dir, "*.[jJ][pP][eE][gG]"))
        )
        self.mask_dir = mask_dir
        self.cache_dir = cache_dir

        self.img_trans = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.mask_trans = T.Compose([
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        name = os.path.splitext(os.path.basename(img_path))[0]

        feat_path = os.path.join(self.cache_dir, f"{name}.pt")
        feat = torch.load(feat_path, weights_only=False).float()

        img = Image.open(img_path).convert("RGB")
        mask_path = os.path.join(self.mask_dir, f"{name}.png")
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.mask_dir, f"{name}.jpg")

        mask = Image.open(mask_path).convert("L")
        return self.img_trans(img), feat, (self.mask_trans(mask) > 0.5).float()

test_dataset = CachedCODDataset(
    img_dir=test_img_dir,
    mask_dir=test_mask_dir,
    cache_dir=test_cache_dir
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f"[✓] Validation Success: test_loader instantiated with {len(test_loader)} images.")

[✓] Validation Success: test_loader instantiated with 4000 images.


In [ ]:
# Cell 1: Cache Test Set Features
import os
import glob
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.transforms as T
from segment_anything import sam_model_registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Dynamically locate test image directory
test_img_dir = None
for root, dirs, files in os.walk("/content/dataset"):
    if root.endswith(os.path.join("Test", "Image")) or root.endswith("Test/Image"):
        test_img_dir = root
        break

if test_img_dir is None:
    test_img_dir = "/content/dataset/COD10K-v3/Test/Image"

test_cache_dir = "/content/cached_feats_test"
os.makedirs(test_cache_dir, exist_ok=True)

test_img_paths = sorted(
    glob.glob(os.path.join(test_img_dir, "*.[jJ][pP][gG]")) +
    glob.glob(os.path.join(test_img_dir, "*.[pP][nN][gG]")) +
    glob.glob(os.path.join(test_img_dir, "*.[jJ][pP][eE][gG]"))
)
print(f"Discovered {len(test_img_paths)} test images at: {test_img_dir}")

# 2. Load SAM ViT-B in evaluation mode
sam_base_path = "/content/sam_vit_b_01ec64.pth"
sam_test = sam_model_registry["vit_b"](checkpoint=sam_base_path).to(device)
sam_test.eval()

transform = T.Compose([
    T.Resize((1024, 1024)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Extracting test embeddings to disk...")
with torch.no_grad():
    for img_path in tqdm(test_img_paths, desc="Caching Test Features"):
        name = os.path.splitext(os.path.basename(img_path))[0]
        out_file = os.path.join(test_cache_dir, f"{name}.pt")
        if os.path.exists(out_file):
            continue

        img = Image.open(img_path).convert("RGB")
        tensor = transform(img).unsqueeze(0).to(device)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            emb = sam_test.image_encoder(tensor)

        torch.save(emb.squeeze(0).half().cpu(), out_file)

# Release the heavy ViT encoder from VRAM
del sam_test
torch.cuda.empty_cache()
print("Test feature caching complete!")

Discovered 4000 test images at: /content/dataset/COD10K-v3/Test/Image
Extracting test embeddings to disk...


Caching Test Features: 100%|██████████| 4000/4000 [17:35<00:00,  3.79it/s]

Test feature caching complete!


In [ ]:
import glob
print(f"Total cached test files: {len(glob.glob('/content/cached_feats_test/*.pt'))}")

Total cached test files: 4000


In [ ]:
# Cell 2: DataLoader & Evaluation Loop
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from py_sod_metrics import Smeasure, Emeasure, WeightedFmeasure, MAE

# 1. Locate test mask directory
test_mask_dir = None
for root, dirs, files in os.walk("/content/dataset"):
    if root.endswith(os.path.join("Test", "GT_Object")) or root.endswith("Test/GT_Object"):
        test_mask_dir = root
        break

if test_mask_dir is None:
    test_mask_dir = "/content/dataset/COD10K-v3/Test/GT_Object"

class CachedCODDataset(Dataset):
    def __init__(self, img_dir, mask_dir, cache_dir, img_size=1024):
        self.img_paths = sorted(
            glob.glob(os.path.join(img_dir, "*.[jJ][pP][gG]")) +
            glob.glob(os.path.join(img_dir, "*.[pP][nN][gG]")) +
            glob.glob(os.path.join(img_dir, "*.[jJ][pP][eE][gG]"))
        )
        self.mask_dir = mask_dir
        self.cache_dir = cache_dir

        self.img_trans = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.mask_trans = T.Compose([
            T.Resize((img_size, img_size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        name = os.path.splitext(os.path.basename(img_path))[0]
        feat = torch.load(os.path.join(self.cache_dir, f"{name}.pt"), weights_only=False).float()

        img = Image.open(img_path).convert("RGB")
        mask_path = os.path.join(self.mask_dir, f"{name}.png")
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.mask_dir, f"{name}.jpg")

        mask = Image.open(mask_path).convert("L")
        return self.img_trans(img), feat, (self.mask_trans(mask) > 0.5).float()

test_dataset = CachedCODDataset(
    img_dir=test_img_dir,
    mask_dir=test_mask_dir,
    cache_dir=test_cache_dir
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f"[✓] DataLoader ready with {len(test_loader)} images.")

# 2. Run Evaluation
def evaluate_ablation_mode(test_loader, model, mode="baseline"):
    s_metric = Smeasure()
    e_metric = Emeasure()
    f_metric = WeightedFmeasure()
    mae_metric = MAE()
    evaluated_count = 0

    with torch.no_grad():
        for imgs, feats, masks in tqdm(test_loader, desc=f"Evaluating {mode}"):
            gt_np = masks.squeeze().numpy().astype(np.uint8)

            # Apples-to-apples protocol: skip non-camouflaged / empty images
            if gt_np.sum() == 0:
                continue

            imgs, feats = imgs.to(device), feats.to(device)

            if mode == "baseline":
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    logits, _ = model(imgs, feats)
                prob = torch.sigmoid(logits).squeeze().cpu().numpy()
            elif mode == "post_proc":
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    logits, _ = model(imgs, feats)
                prob_raw = torch.sigmoid(logits).squeeze().cpu().numpy()
                prob = post_process_mask(prob_raw, threshold=0.5)
            elif mode == "self_prompt":
                prob = self_prompted_forward(model, imgs, feats, pad_ratio=0.05)
            elif mode == "self_prompt_post_proc":
                prob_raw = self_prompted_forward(model, imgs, feats, pad_ratio=0.05)
                prob = post_process_mask(prob_raw, threshold=0.5)

            pred_255 = (prob * 255).astype(np.uint8)
            gt_255 = (gt_np * 255).astype(np.uint8)

            s_metric.step(pred_255, gt_255)
            e_metric.step(pred_255, gt_255)
            f_metric.step(pred_255, gt_255)
            mae_metric.step(pred_255, gt_255)
            evaluated_count += 1

    sm = s_metric.get_results()['sm']
    em = e_metric.get_results()['em']['curve'].mean()
    wfm = f_metric.get_results()['wfm']
    mae = mae_metric.get_results()['mae']

    print(f"\n================= Results: {mode} ({evaluated_count} Positive Images) =================")
    print(f"  Structure Measure (S_alpha)       : {sm:.4f}")
    print(f"  Enhanced Alignment (E_phi mean)   : {em:.4f}")
    print(f"  Weighted F-measure (F_beta^w)     : {wfm:.4f}")
    print(f"  Mean Absolute Error (MAE)         : {mae:.4f}")
    print("========================================================================================\n")
    return {"mode": mode, "Sm": sm, "Em": em, "wFm": wfm, "MAE": mae}

baseline_results = evaluate_ablation_mode(test_loader, model, mode="baseline")

[✓] DataLoader ready with 4000 images.


Evaluating baseline: 100%|██████████| 4000/4000 [14:02<00:00,  4.75it/s]


================= Results: baseline (2026 Positive Images) =================
  Structure Measure (S_alpha)       : 0.7415
  Enhanced Alignment (E_phi mean)   : 0.7730
  Weighted F-measure (F_beta^w)     : 0.5388
  Mean Absolute Error (MAE)         : 0.0624



In [ ]:
# Cell: Run Zero-Retraining Inference Ablations
print(">>> Evaluating Mode 2: Baseline + Morphological Post-Processing")
res_post = evaluate_ablation_mode(test_loader, model, mode="post_proc")

print("\n>>> Evaluating Mode 3: Baseline + Two-Pass Self-Prompting")
res_prompt = evaluate_ablation_mode(test_loader, model, mode="self_prompt")

print("\n>>> Evaluating Mode 4: Baseline + Self-Prompting + Post-Processing")
res_final = evaluate_ablation_mode(test_loader, model, mode="self_prompt_post_proc")

>>> Evaluating Mode 2: Baseline + Morphological Post-Processing


Evaluating post_proc: 100%|██████████| 4000/4000 [16:09<00:00,  4.13it/s]



================= Results: post_proc (2026 Positive Images) =================
  Structure Measure (S_alpha)       : 0.7290
  Enhanced Alignment (E_phi mean)   : 0.8057
  Weighted F-measure (F_beta^w)     : 0.5758
  Mean Absolute Error (MAE)         : 0.0552


>>> Evaluating Mode 3: Baseline + Two-Pass Self-Prompting


Evaluating self_prompt: 100%|██████████| 4000/4000 [13:43<00:00,  4.86it/s]



================= Results: self_prompt (2026 Positive Images) =================
  Structure Measure (S_alpha)       : 0.7323
  Enhanced Alignment (E_phi mean)   : 0.7742
  Weighted F-measure (F_beta^w)     : 0.5188
  Mean Absolute Error (MAE)         : 0.0694


>>> Evaluating Mode 4: Baseline + Self-Prompting + Post-Processing


Evaluating self_prompt_post_proc: 100%|██████████| 4000/4000 [16:25<00:00,  4.06it/s]


================= Results: self_prompt_post_proc (2026 Positive Images) =================
  Structure Measure (S_alpha)       : 0.7199
  Enhanced Alignment (E_phi mean)   : 0.8008
  Weighted F-measure (F_beta^w)     : 0.5562
  Mean Absolute Error (MAE)         : 0.0618



In [ ]:
# Cell 1: Architecture & Loss Function for Retraining
import torch
import torch.nn as nn
import torch.nn.functional as F
from segment_anything import sam_model_registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sam_base_path = "/content/sam_vit_b_01ec64.pth"

# 1. Sobel Edge Detector (Fixed buffers)
class SobelEdgeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, x):
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        gx = F.conv2d(gray, self.sobel_x, padding=1)
        gy = F.conv2d(gray, self.sobel_y, padding=1)
        edges = torch.sqrt(gx**2 + gy**2 + 1e-6)
        return (edges - edges.min()) / (edges.max() - edges.min() + 1e-6)

# 2. Learnable Multi-Scale Edge Pyramid Adapter
class EdgePyramidAdapter(nn.Module):
    def __init__(self, in_dim=256, bottleneck=64):
        super().__init__()
        self.down = nn.Conv2d(in_dim, bottleneck, kernel_size=1)

        def blk(i, o, s):
            return nn.Sequential(
                nn.Conv2d(i, o, kernel_size=3, stride=s, padding=1),
                nn.BatchNorm2d(o),
                nn.GELU()
            )

        # Progressive downsampling: 1024 -> 512 -> 256 -> 128 -> 64
        self.s1 = blk(1, 16, 2)
        self.s2 = blk(16, 32, 2)
        self.s3 = blk(32, 48, 2)
        self.s4 = blk(48, bottleneck, 2)

        self.fuse = nn.Sequential(
            nn.Conv2d(bottleneck * 2, bottleneck, kernel_size=3, padding=1),
            nn.BatchNorm2d(bottleneck),
            nn.GELU()
        )
        self.up = nn.Conv2d(bottleneck, in_dim, kernel_size=1)

        # Zero-init final projection so initial forward pass matches current model
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, feat, edge_map):
        edge_feat = self.s4(self.s3(self.s2(self.s1(edge_map))))
        feat_down = self.down(feat)
        fused = self.fuse(torch.cat([feat_down, edge_feat], dim=1))
        return feat + self.up(fused)

# 3. Full Adapted SAM Model
class EdgePyramidSAM(nn.Module):
    def __init__(self, sam_checkpoint):
        super().__init__()
        base_sam = sam_model_registry["vit_b"](checkpoint=sam_checkpoint)
        self.edge_detector = SobelEdgeDetector()
        self.adapter = EdgePyramidAdapter(in_dim=256, bottleneck=64)
        self.prompt_encoder = base_sam.prompt_encoder
        self.mask_decoder = base_sam.mask_decoder

        for p in self.prompt_encoder.parameters():
            p.requires_grad = False
        for p in self.mask_decoder.parameters():
            p.requires_grad = True

    def forward(self, imgs, cached_feats):
        edge_map = self.edge_detector(imgs)
        refined_feats = self.adapter(cached_feats, edge_map)
        sparse_pe, dense_pe = self.prompt_encoder(points=None, boxes=None, masks=None)
        low_res, _ = self.mask_decoder(
            image_embeddings=refined_feats,
            image_pe=self.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_pe,
            dense_prompt_embeddings=dense_pe,
            multimask_output=False,
        )
        return F.interpolate(low_res, size=(1024, 1024), mode="bilinear", align_corners=False), edge_map

# 4. Upgraded Boundary-Aware Loss (Both Horizontal & Vertical Gradients)
class BoundaryAwareLoss(nn.Module):
    def __init__(self, edge_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.edge_weight = edge_weight

    def forward(self, pred_mask, target_mask):
        bce = self.bce(pred_mask, target_mask)
        prob = torch.sigmoid(pred_mask)

        # Region IoU
        inter = (prob * target_mask).sum(dim=(2, 3))
        union = (prob + target_mask).sum(dim=(2, 3)) - inter
        iou = (1.0 - (inter + 1.0) / (union + 1.0)).mean()

        # Dual-Directional Edge Gradients (Horizontal + Vertical)
        p_gx = torch.abs(prob[:, :, :, :-1] - prob[:, :, :, 1:])
        t_gx = torch.abs(target_mask[:, :, :, :-1] - target_mask[:, :, :, 1:])
        p_gy = torch.abs(prob[:, :, :-1, :] - prob[:, :, 1:, :])
        t_gy = torch.abs(target_mask[:, :, :-1, :] - target_mask[:, :, 1:, :])

        edge_loss = F.l1_loss(p_gx, t_gx) + F.l1_loss(p_gy, t_gy)
        return bce + iou + self.edge_weight * edge_loss

print("[✓] Architecture & Loss defined successfully.")

[✓] Architecture & Loss defined successfully.


In [ ]:
# Cell 2: Warm-Start Initialization
model_pyramid = EdgePyramidSAM(sam_base_path).to(device)

# Load existing checkpoint weights
ckpt_path = "/content/drive/MyDrive/COD_Checkpoints/edge_sam_best.pth"
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt

# Filter and load matching weights (strict=False adapts to the new pyramid layers)
missing, unexpected = model_pyramid.load_state_dict(state_dict, strict=False)
print(f"[✓] Initialized from previous best checkpoint.")
print(f"    New pyramid layers initialized cleanly: {[k for k in missing if 's' in k or 'fuse' in k][:4]}...")

[✓] Initialized from previous best checkpoint.
    New pyramid layers initialized cleanly: ['adapter.s1.0.weight', 'adapter.s1.0.bias', 'adapter.s1.1.weight', 'adapter.s1.1.bias']...


In [ ]:
!ls -lh /content
!ls -lh /content/cached_feats_test | head -n 5
!ls -lh /content/cached_feats | head -n 5

total 358M
drwxr-xr-x 2 root root 264K Oct 13 19:27 cached_feats_test
drwxr-xr-x 3 root root 4.0K Oct 13 19:06 dataset
drwx------ 5 root root 4.0K Oct 13 18:59 drive
drwxr-xr-x 1 root root 4.0K Sep 30 13:28 sample_data
-rw-r--r-- 1 root root 358M Oct 13 19:00 sam_vit_b_01ec64.pth
total 7.9G
-rw-r--r-- 1 root root 2.1M Oct 13 19:10 COD10K-CAM-1-Aquatic-10-LeafySeaDragon-416.pt
-rw-r--r-- 1 root root 2.1M Oct 13 19:10 COD10K-CAM-1-Aquatic-10-LeafySeaDragon-419.pt
-rw-r--r-- 1 root root 2.1M Oct 13 19:10 COD10K-CAM-1-Aquatic-10-LeafySeaDragon-422.pt
-rw-r--r-- 1 root root 2.1M Oct 13 19:10 COD10K-CAM-1-Aquatic-10-LeafySeaDragon-423.pt
ls: cannot access '/content/cached_feats': No such file or directory


In [ ]:
import os
from google.colab import drive

# 1. Ensure Google Drive is mounted
drive.mount('/content/drive')

# 2. Setup backup directory in your Drive
backup_dir = "/content/drive/MyDrive/COD_Backup"
os.makedirs(backup_dir, exist_ok=True)
print(f"[✓] Backup directory ready: {backup_dir}")

# 3. Archive and save Test Cached Features (~4,000 files)
if os.path.exists("/content/cached_feats_test") and len(os.listdir("/content/cached_feats_test")) > 0:
    print("Archiving test features (/content/cached_feats_test)...")
    !tar -czf {backup_dir}/cached_feats_test.tar.gz -C /content cached_feats_test
    print("Saved cached_feats_test.tar.gz to Google Drive!")
else:
    print("cached_feats_test folder is empty or missing, skipping.")

# 4. Archive and save Training Cached Features (~6,000 files)
if os.path.exists("/content/cached_feats") and len(os.listdir("/content/cached_feats")) > 0:
    print("Archiving train features (/content/cached_feats)...")
    !tar -czf {backup_dir}/cached_feats_train.tar.gz -C /content cached_feats
    print("Saved cached_feats_train.tar.gz to Google Drive!")
else:
    print("cached_feats folder is empty or missing, skipping.")

# 5. Backup Checkpoints folder
if os.path.exists("/content/checkpoints"):
    print("Backing up checkpoints...")
    !cp -r /content/checkpoints {backup_dir}/
    print("Checkpoints backed up!")

print("\n==========================================")
print("All current progress safely stored in Google Drive!")
!ls -lh {backup_dir}
print("==========================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[✓] Backup directory ready: /content/drive/MyDrive/COD_Backup
Archiving test features (/content/cached_feats_test)...
Saved cached_feats_test.tar.gz to Google Drive!
cached_feats folder is empty or missing, skipping.

All current progress safely stored in Google Drive!
total 7.0G
-rw------- 1 root root 7.0G Oct 13 20:50 cached_feats_test.tar.gz


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Restore cached features in under 30 seconds
!tar -xzf /content/drive/MyDrive/COD_Backup/cached_feats_test.tar.gz -C /content/
!tar -xzf /content/drive/MyDrive/COD_Backup/cached_feats_train.tar.gz -C /content/

print("Restoration complete! Features ready in /content/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tar (child): /content/drive/MyDrive/COD_Backup/cached_feats_train.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
Restoration complete! Features ready in /content/


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

backup_dir = "/content/drive/MyDrive/COD_Backup"
os.makedirs(backup_dir, exist_ok=True)

print("Archiving /content/cached_feats_test to Google Drive...")
!tar -czf {backup_dir}/cached_feats_test.tar.gz -C /content cached_feats_test
print("[✓] Validation: cached_feats_test.tar.gz saved to Google Drive!")
!ls -lh {backup_dir}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archiving /content/cached_feats_test to Google Drive...
^C
[✓] Validation: cached_feats_test.tar.gz saved to Google Drive!
total 5.6G
-rw------- 1 root root 5.6G Oct 13 20:59 cached_feats_test.tar.gz


In [ ]:
import os
import glob
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.transforms as T
from segment_anything import sam_model_registry

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Locate training images dynamically
train_img_dir = None
for root, dirs, files in os.walk("/content/dataset"):
    if root.endswith(os.path.join("Train", "Image")) or root.endswith("Train/Image"):
        train_img_dir = root
        break

if train_img_dir is None:
    train_img_dir = "/content/dataset/COD10K-v3/Train/Image"

train_cache_dir = "/content/cached_feats"
os.makedirs(train_cache_dir, exist_ok=True)

train_img_paths = sorted(
    glob.glob(os.path.join(train_img_dir, "*.[jJ][pP][gG]")) +
    glob.glob(os.path.join(train_img_dir, "*.[pP][nN][gG]")) +
    glob.glob(os.path.join(train_img_dir, "*.[jJ][pP][eE][gG]"))
)
print(f"Found {len(train_img_paths)} train images at: {train_img_dir}")
assert len(train_img_paths) > 0, "No train images found in /content/dataset!"

# 2. Extract features using frozen SAM ViT
sam_base_path = "/content/sam_vit_b_01ec64.pth"
sam_extractor = sam_model_registry["vit_b"](checkpoint=sam_base_path).to(device)
sam_extractor.eval()

transform = T.Compose([
    T.Resize((1024, 1024)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Extracting training embeddings to disk...")
with torch.no_grad():
    for img_path in tqdm(train_img_paths, desc="Caching Train Features"):
        name = os.path.splitext(os.path.basename(img_path))[0]
        out_file = os.path.join(train_cache_dir, f"{name}.pt")
        if os.path.exists(out_file):
            continue

        img = Image.open(img_path).convert("RGB")
        tensor = transform(img).unsqueeze(0).to(device)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            emb = sam_extractor.image_encoder(tensor)

        torch.save(emb.squeeze(0).half().cpu(), out_file)

# Free VRAM
del sam_extractor
torch.cuda.empty_cache()

cached_count = len(glob.glob(os.path.join(train_cache_dir, "*.pt")))
print(f"\n[✓] Train feature extraction complete! Total files in /content/cached_feats: {cached_count}")

Found 6000 train images at: /content/dataset/COD10K-v3/Train/Image
Extracting training embeddings to disk...


Caching Train Features: 100%|██████████| 6000/6000 [25:22<00:00,  3.94it/s]


[✓] Train feature extraction complete! Total files in /content/cached_feats: 6000


In [ ]:
print("Archiving /content/cached_feats to Google Drive...")
!tar -czf {backup_dir}/cached_feats_train.tar.gz -C /content cached_feats
print("[✓] Validation: cached_feats_train.tar.gz saved to Google Drive!")
!ls -lh {backup_dir}

Archiving /content/cached_feats to Google Drive...
[✓] Validation: cached_feats_train.tar.gz saved to Google Drive!
total 16G
-rw------- 1 root root 5.6G Oct 13 20:59 cached_feats_test.tar.gz
-rw------- 1 root root  11G Oct 13 21:36 cached_feats_train.tar.gz


In [ ]:
# Cell: Edge Pyramid Training Loop
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. Verify train directories
train_img_dir = "/content/dataset/COD10K-v3/Train/Image"
train_mask_dir = "/content/dataset/COD10K-v3/Train/GT_Object"
train_cache_dir = "/content/cached_feats"

# Fallback path check if paths are nested differently
if not os.path.exists(train_img_dir):
    for root, dirs, files in os.walk("/content/dataset"):
        if root.endswith("Train/Image") or root.endswith(os.path.join("Train", "Image")):
            train_img_dir = root
        if root.endswith("Train/GT_Object") or root.endswith(os.path.join("Train", "GT_Object")):
            train_mask_dir = root

assert os.path.exists(train_cache_dir), f"Missing cached train features at {train_cache_dir}!"

# 2. Dataset & DataLoader (Using batch_size=4 to maintain ~1.2 GB VRAM footprint)
train_dataset = CachedCODDataset(train_img_dir, train_mask_dir, train_cache_dir)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# 3. Optimization Setup
criterion = BoundaryAwareLoss(edge_weight=0.5).to(device)
trainable_params = [p for p in model_pyramid.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6, eta_min=1e-6)

save_dir = "/content/drive/MyDrive/COD_Checkpoints"
os.makedirs(save_dir, exist_ok=True)
best_loss = float("inf")

print(f"Starting 6-Epoch Training ({len(train_loader)} batches/epoch)...")

# 4. Training Loop
for epoch in range(1, 7):
    model_pyramid.train()
    running_loss = 0.0

    for step, (imgs, feats, masks) in enumerate(train_loader):
        imgs = imgs.to(device)
        feats = feats.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred_masks, _ = model_pyramid(imgs, feats)
            loss = criterion(pred_masks, masks)

        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if (step + 1) % 300 == 0 or (step + 1) == len(train_loader):
            print(f"Epoch [{epoch}/6] | Step [{step+1}/{len(train_loader)}] | Step Loss: {loss.item():.4f}")

    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    lr_curr = optimizer.param_groups[0]['lr']
    print(f"---> Epoch {epoch} Completed | Avg Loss: {avg_loss:.4f} | LR: {lr_curr:.6f}")

    # Checkpoint saving
    if avg_loss < best_loss:
        best_loss = avg_loss
        save_path = f"{save_dir}/edge_pyramid_sam_best.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pyramid.state_dict(),
            'loss': best_loss
        }, save_path)
        print(f"  [*] Saved new best model checkpoint to: {save_path}")

print("\nTraining completed successfully!")

Starting 6-Epoch Training (1500 batches/epoch)...
Epoch [1/6] | Step [300/1500] | Step Loss: 1.3145
Epoch [1/6] | Step [600/1500] | Step Loss: 0.8322
Epoch [1/6] | Step [900/1500] | Step Loss: 0.9643
Epoch [1/6] | Step [1200/1500] | Step Loss: 1.0390
Epoch [1/6] | Step [1500/1500] | Step Loss: 1.0818
---> Epoch 1 Completed | Avg Loss: 1.0301 | LR: 0.000093
  [*] Saved new best model checkpoint to: /content/drive/MyDrive/COD_Checkpoints/edge_pyramid_sam_best.pth
Epoch [2/6] | Step [300/1500] | Step Loss: 0.7932
Epoch [2/6] | Step [600/1500] | Step Loss: 1.0366
Epoch [2/6] | Step [900/1500] | Step Loss: 0.5825
Epoch [2/6] | Step [1200/1500] | Step Loss: 0.9010
Epoch [2/6] | Step [1500/1500] | Step Loss: 1.1492
---> Epoch 2 Completed | Avg Loss: 0.9664 | LR: 0.000075
  [*] Saved new best model checkpoint to: /content/drive/MyDrive/COD_Checkpoints/edge_pyramid_sam_best.pth
Epoch [3/6] | Step [300/1500] | Step Loss: 1.1415
Epoch [3/6] | Step [600/1500] | Step Loss: 0.9761
Epoch [3/6] | Step

In [ ]:
# ==============================================================================
# AUTOMATIC OVERNIGHT SAVE, EVALUATION, AND AUTO-SHUTDOWN SCRIPT
# ==============================================================================
import os
import glob
import json
import torch
import numpy as np
from tqdm import tqdm
from google.colab import drive, runtime
from py_sod_metrics import Smeasure, Emeasure, WeightedFmeasure, MAE

# 1. Verify Drive Mount
drive.mount('/content/drive')
checkpoint_dir = "/content/drive/MyDrive/COD_Checkpoints"
backup_dir = "/content/drive/MyDrive/COD_Backup"
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(backup_dir, exist_ok=True)

print("="*60)
print("[1/4] Securing Final Model Checkpoints & Features to Drive...")
print("="*60)

# Save the final in-memory model weights just in case
final_save_path = os.path.join(checkpoint_dir, "edge_pyramid_sam_final.pth")
torch.save({
    'model_state_dict': model_pyramid.state_dict(),
}, final_save_path)
print(f"[✓] Final model state preserved at: {final_save_path}")

# Archive training features to Drive if not already done
train_tar = os.path.join(backup_dir, "cached_feats_train.tar.gz")
if os.path.exists("/content/cached_feats") and not os.path.exists(train_tar):
    print("Archiving /content/cached_feats to Google Drive...")
    !tar -czf {train_tar} -C /content cached_feats
    print(f"[✓] Saved {train_tar}")
else:
    print("[✓] Train features already backed up or directory absent.")

print("\n" + "="*60)
print("[2/4] Running Standardized COD Evaluation on 2,026 Positive Images...")
print("="*60)

# Make sure best checkpoint is loaded for evaluation
best_ckpt_path = os.path.join(checkpoint_dir, "edge_pyramid_sam_best.pth")
if not os.path.exists(best_ckpt_path):
    best_ckpt_path = final_save_path

print(f"Loading weights from: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model_pyramid.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
model_pyramid.eval()

# Evaluator setup
s_metric = Smeasure()
e_metric = Emeasure()
f_metric = WeightedFmeasure()
mae_metric = MAE()
pos_count = 0

with torch.no_grad():
    for imgs, feats, masks in tqdm(test_loader, desc="Overnight Evaluation"):
        gt_np = masks.squeeze().numpy().astype(np.uint8)

        # Apples-to-apples COD protocol: evaluate foreground targets only
        if gt_np.sum() == 0:
            continue

        imgs, feats = imgs.to(device), feats.to(device)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits, _ = model_pyramid(imgs, feats)

        prob = torch.sigmoid(logits).squeeze().cpu().numpy()
        pred_255 = (prob * 255).astype(np.uint8)
        gt_255 = (gt_np * 255).astype(np.uint8)

        s_metric.step(pred_255, gt_255)
        e_metric.step(pred_255, gt_255)
        f_metric.step(pred_255, gt_255)
        mae_metric.step(pred_255, gt_255)
        pos_count += 1

# Calculate final metrics
sm = float(s_metric.get_results()['sm'])
em = float(e_metric.get_results()['em']['curve'].mean())
wfm = float(f_metric.get_results()['wfm'])
mae = float(mae_metric.get_results()['mae'])

results_summary = {
    "images_evaluated": pos_count,
    "S_alpha": sm,
    "E_phi_mean": em,
    "Weighted_F_beta": wfm,
    "MAE": mae
}

print("\n" + "="*60)
print("[3/4] Writing Official Results to Google Drive...")
print("="*60)

log_file_txt = os.path.join(checkpoint_dir, "training_and_evaluation_results.txt")
log_file_json = os.path.join(checkpoint_dir, "training_and_evaluation_results.json")

report_text = f"""
================================================================
   COD10K BENCHMARK EVALUATION REPORT (OVERNIGHT RUN)
================================================================
Checkpoint Used   : {best_ckpt_path}
Images Evaluated  : {pos_count} (Positive Camouflaged Targets)
----------------------------------------------------------------
Structure Measure (S_alpha)       : {sm:.4f}
Mean Enhanced Alignment (E_phi)   : {em:.4f}
Weighted F-measure (F_beta^w)     : {wfm:.4f}
Mean Absolute Error (MAE)         : {mae:.4f}
================================================================
"""

with open(log_file_txt, "w") as f:
    f.write(report_text)

with open(log_file_json, "w") as f:
    json.dump(results_summary, f, indent=4)

print(report_text)
print(f"[✓] Log saved to: {log_file_txt}")
print(f"[✓] JSON saved to: {log_file_json}")

print("\n" + "="*60)
print("[4/4] Safely Disconnecting Colab Runtime...")
print("="*60)

# Automatically shut down session to prevent idle compute unit consumption
runtime.unassign()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/4] Securing Final Model Checkpoints & Features to Drive...
[✓] Final model state preserved at: /content/drive/MyDrive/COD_Checkpoints/edge_pyramid_sam_final.pth
[✓] Train features already backed up or directory absent.

[2/4] Running Standardized COD Evaluation on 2,026 Positive Images...
Loading weights from: /content/drive/MyDrive/COD_Checkpoints/edge_pyramid_sam_best.pth


Overnight Evaluation: 100%|██████████| 4000/4000 [13:19<00:00,  5.01it/s]



[3/4] Writing Official Results to Google Drive...

   COD10K BENCHMARK EVALUATION REPORT (OVERNIGHT RUN)
Checkpoint Used   : /content/drive/MyDrive/COD_Checkpoints/edge_pyramid_sam_best.pth
Images Evaluated  : 2026 (Positive Camouflaged Targets)
----------------------------------------------------------------
Structure Measure (S_alpha)       : 0.7213
Mean Enhanced Alignment (E_phi)   : 0.7439
Weighted F-measure (F_beta^w)     : 0.4931
Mean Absolute Error (MAE)         : 0.0684

[✓] Log saved to: /content/drive/MyDrive/COD_Checkpoints/training_and_evaluation_results.txt
[✓] JSON saved to: /content/drive/MyDrive/COD_Checkpoints/training_and_evaluation_results.json

[4/4] Safely Disconnecting Colab Runtime...
